Run the command below in a PyCharm WSL window to access the Python virtual environment:

```shell
 source /home/penguini/.virtualenvs/DrChanWorkPlayground/bin/activate
```

In [1]:
import tensorflow as tf

# List available physical devices
physical_devices = tf.config.list_physical_devices('GPU')
print("Num GPUs Available: ", len(physical_devices))

# If GPUs are found, check if TensorFlow is built with CUDA/ROCm support
if len(physical_devices) > 0:
    print("TensorFlow is built with GPU support:", tf.test.is_built_with_cuda())

2025-08-27 10:49:02.804610: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-27 10:49:02.880900: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-08-27 10:49:04.990483: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Num GPUs Available:  0


2025-08-27 10:49:06.294626: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


In [2]:
import numpy as np
import keras
from keras import layers
from tensorflow.keras.datasets import fashion_mnist
from imbal.metrics import CSI, TPR, ExpectedTP, ExpectedTN, ExpectedCorrect, TNR, FPR, Precision, TSS, HSS, GS

(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

print(y_train[:50])
y_train = np.array([1 if y_train[i] == 0 else 0 for i in range(len(y_train))])
print(y_train[:50])

y_test = np.array([1 if y_test[i] == 0 else 0 for i in range(len(y_test))])

[9 0 0 3 0 2 7 2 5 5 0 9 5 5 7 9 1 0 6 4 3 1 4 8 4 3 0 2 4 4 5 3 6 6 0 8 5
 2 1 6 6 7 9 5 9 2 7 3 0 3]
[0 1 1 0 1 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0
 0 0 0 0 0 0 0 0 0 0 0 1 0]


In [3]:
inputs = keras.Input(shape=(28,28))
flatten = layers.Flatten()(inputs)
hidden = layers.Dense(64, activation='relu')(flatten)
output = layers.Dense(1, activation='sigmoid')(hidden)

model = keras.Model(inputs=inputs, outputs=output)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=["accuracy", TPR(), FPR(), TNR(), Precision, ExpectedTP(), ExpectedTN(), ExpectedCorrect(), TSS(), HSS(), GS(), CSI()])

In [4]:
history = model.fit(x_train, y_train, batch_size=256, epochs=5)

Epoch 1/5
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9336 - csi: 0.5044 - expected_correct: 49119.1992 - expected_tn: 48509.0977 - expected_tp: 610.1000 - fpr: 0.0379 - gs: 0.4637 - hss: 0.6336 - loss: 2.6318 - precision: 0.6650 - tnr: 0.9621 - tpr: 0.6762 - tss: 0.6383
Epoch 2/5
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9453 - csi: 0.5702 - expected_correct: 49204.0000 - expected_tn: 48604.5000 - expected_tp: 599.5000 - fpr: 0.0304 - gs: 0.5336 - hss: 0.6959 - loss: 0.4177 - precision: 0.7266 - tnr: 0.9696 - tpr: 0.7260 - tss: 0.6956
Epoch 3/5
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9489 - csi: 0.5916 - expected_correct: 49235.1992 - expected_tn: 48639.5977 - expected_tp: 595.6000 - fpr: 0.0280 - gs: 0.5564 - hss: 0.7150 - loss: 0.2870 - precision: 0.7461 - tnr: 0.9720 - tpr: 0.7407 - tss: 0.7127
Epoch 4/5
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9520 - csi: 0.6102 - expected_correct: 49276.7969 - expected_tn: 48686.3984 - expected

In [5]:
test_scores = model.evaluate(x_test, y_test, verbose=1, batch_size=250)
print("Test loss:", test_scores[0])
print(x_test.shape[0])

40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9516 - csi: 0.5713 - expected_correct: 8380.8008 - expected_tn: 8303.4004 - expected_tp: 77.4000 - fpr: 0.0143 - gs: 0.5397 - hss: 0.7011 - loss: 0.1655 - precision: 0.8333 - tnr: 0.9857 - tpr: 0.6450 - tss: 0.6307
Test loss: 0.16545628011226654
10000


In [6]:
print(model.predict(np.reshape(x_test[12], (1, 28, 28))))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
[[3.7947226e-21]]


In [7]:
print("Test loss:", test_scores[0])
print("Test accuracy:", test_scores[1])
print("Test TPR:", test_scores[2])
print("Test FPR:", test_scores[3])
print("Test TNR:", test_scores[4])
print("Test Precision:", test_scores[5])
print("Test Expected TP:", test_scores[6])
print("Test Expected TN:", test_scores[7])
print("Test Expected Correct:", test_scores[8])
print("Test TSS:", test_scores[9])
print("Test HSS:", test_scores[10])
print("Test GS:", test_scores[11])
print("Test CSI:", test_scores[12])

Test loss: 0.16545628011226654
Test accuracy: 0.9516000151634216
Test TPR: 0.6449999809265137
Test FPR: 0.014333332888782024
Test TNR: 0.9856666922569275
Test Precision: 0.8333333134651184
Test Expected TP: 77.4000015258789
Test Expected TN: 8303.400390625
Test Expected Correct: 8380.80078125
Test TSS: 0.6306666731834412
Test HSS: 0.7010868191719055
Test GS: 0.5397489666938782
Test CSI: 0.571302056312561


In [ ]:
print(history)

In [ ]:
import matplotlib.pyplot as plt

labels = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot"
]

data_index = 8


plt.imshow(x_test[data_index], cmap=plt.get_cmap('gray'))
print(labels[np.argmax(model.predict(np.reshape(x_test[data_index], (1, 28, 28))))])


In [ ]:
import tensorflow as tf

# List available physical devices
physical_devices = tf.config.list_physical_devices('GPU')
print("Num GPUs Available: ", len(physical_devices))

# If GPUs are found, check if TensorFlow is built with CUDA/ROCm support
if len(physical_devices) > 0:
    print("TensorFlow is built with GPU support:", tf.test.is_built_with_cuda())

In [ ]:
import numpy as np
import keras
from keras import layers
from tensorflow.keras.datasets import mnist

(x_train, y_train), (x_test, y_test) = mnist.load_data()

In [ ]:
inputs = keras.Input(shape=(28,28))
flatten = layers.Flatten()(inputs)
hidden = layers.Dense(64, activation='relu')(flatten)
output = layers.Dense(10, activation='softmax')(hidden)

model = keras.Model(inputs=inputs, outputs=output)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
history = model.fit(x_train, y_train, batch_size=256, epochs=20, validation_data=(x_test, y_test))

In [ ]:
test_scores = model.evaluate(x_test, y_test, verbose=1)

In [ ]:
print("Test loss:", test_scores[0])
print("Test accuracy:", test_scores[1])

In [ ]:
print(history)

In [ ]:
import matplotlib.pyplot as plt

data_index = 12


plt.imshow(x_test[data_index], cmap=plt.get_cmap('gray'))
print(np.argmax(model.predict(np.reshape(x_test[data_index], (1, 28, 28)))))
